In [ ]:
# ================================================================
# CELL S1: INSTALLS
# ================================================================
!pip install -q transformers accelerate bitsandbytes sentencepiece
!pip install -q sentence-transformers scikit-learn scipy matplotlib seaborn
!pip install -q huggingface_hub

In [ ]:


# ================================================================
# CELL S2: IMPORTS & SEEDS
# ================================================================
import os, json, pickle, warnings
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats as scipy_stats
from scipy.linalg import sqrtm
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    roc_curve, brier_score_loss,
    precision_recall_curve,
)
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    AutoModelForSequenceClassification,
    BitsAndBytesConfig, set_seed,
    TrainingArguments, Trainer, EarlyStoppingCallback,
)
from torch.utils.data import Dataset, DataLoader
from sentence_transformers import SentenceTransformer
from tqdm import tqdm
warnings.filterwarnings("ignore")

SEED = 42
set_seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

for d in ["./stage3_results", "./stage3_results/3A", "./stage3_results/3B",
          "./stage3_results/3C", "./stage3_results/3D", "./stage3_results/3E"]:
    os.makedirs(d, exist_ok=True)

print("✅ Imports ready")

# ================================================================
# CELL S3: MODEL REGISTRY
# ================================================================
MODEL_REGISTRY = {
    "TinyLlama-1.1B": {
        "hf_id":        "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
        "gated":        False,
        "use_4bit":     False,
        "prompt_style": "chatml",
    },
    "Qwen2.5-1.5B": {
        "hf_id":        "Qwen/Qwen2.5-1.5B-Instruct",
        "gated":        False,
        "use_4bit":     False,
        "prompt_style": "chatml",
    },
    "Qwen2.5-7B": {
        "hf_id":        "Qwen/Qwen2.5-7B-Instruct",
        "gated":        False,
        "use_4bit":     True,
        "prompt_style": "chatml",
    },
    "LLaMA-3.1-8B": {
        "hf_id":        "meta-llama/Meta-Llama-3.1-8B-Instruct",
        "gated":        True,
        "use_4bit":     True,
        "prompt_style": "llama3",
    },
    "LLaMA-2-13B": {
        "hf_id":        "meta-llama/Llama-2-13b-chat-hf",
        "gated":        True,
        "use_4bit":     True,
        "prompt_style": "llama2",
    },
    # "Qwen2.5-14B": {
    #     "hf_id":        "Qwen/Qwen2.5-14B-Instruct",
    #     "gated":        False,
    #     "use_4bit":     True,
    #     "prompt_style": "chatml",
    # },
}

N_SAMPLES      = 200
MAX_NEW_TOKENS = 150
N_BOOT         = 1000

print("Model registry loaded:")
for name, cfg in MODEL_REGISTRY.items():
    print(f"  {name:20s}  4bit={cfg['use_4bit']}  gated={cfg['gated']}  style={cfg['prompt_style']}")


# ================================================================
# CELL G0: MODEL LOADING + GENERATION HELPERS
# ================================================================
from google.colab import userdata
from huggingface_hub import login

HF_TOKEN = userdata.get("HF_TOKEN")
login(token=HF_TOKEN, add_to_git_credential=False)
print("✅ Authenticated to HuggingFace Hub")

SYSTEM_PROMPT = (
    "You are a helpful assistant. Answer the question clearly "
    "and concisely in 3-5 sentences."
)


def format_prompt(user_text, prompt_style, tokenizer):
    """
    Format a raw user question into the correct prompt format
    for each model family. Uses apply_chat_template where
    available, falls back to manual string formatting.
    """
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": user_text},
    ]

    if prompt_style == "chatml":

        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

    elif prompt_style == "llama3":

        if hasattr(tokenizer, "apply_chat_template") and tokenizer.chat_template:
            return tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
            )
        else:
            return (
                f"<|begin_of_text|>"
                f"<|start_header_id|>system<|end_header_id|>\n\n"
                f"{SYSTEM_PROMPT}<|eot_id|>"
                f"<|start_header_id|>user<|end_header_id|>\n\n"
                f"{user_text}<|eot_id|>"
                f"<|start_header_id|>assistant<|end_header_id|>\n\n"
            )

    elif prompt_style == "llama2":

        return (
            f"<s>[INST] <<SYS>>\n{SYSTEM_PROMPT}\n<</SYS>>\n\n"
            f"{user_text} [/INST]"
        )

    else:
        raise ValueError(f"Unknown prompt_style: {prompt_style}")


def load_causal_model(model_cfg):
    """
    Load a causal LM with optional 4-bit quantization.
    Handles tokenizer padding side correctly for generation.
    """
    hf_id    = model_cfg["hf_id"]
    use_4bit = model_cfg["use_4bit"]

    print(f"  Loading tokenizer : {hf_id}")
    tokenizer = AutoTokenizer.from_pretrained(
        hf_id,
        token=HF_TOKEN,
        trust_remote_code=True,
    )

    tokenizer.padding_side = "left"

    # Some tokenizers have no pad token — use eos as fallback
    if tokenizer.pad_token is None:
        tokenizer.pad_token    = tokenizer.eos_token
        tokenizer.pad_token_id = tokenizer.eos_token_id

    print(f"  Loading model     : {hf_id}  (4bit={use_4bit})")

    if use_4bit:
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        )
        model = AutoModelForCausalLM.from_pretrained(
            hf_id,
            quantization_config=bnb_config,
            device_map="auto",
            token=HF_TOKEN,
            trust_remote_code=True,
        )
    else:
        model = AutoModelForCausalLM.from_pretrained(
            hf_id,
            dtype=torch.float16,
            device_map="auto",
            token=HF_TOKEN,
            trust_remote_code=True,
        )

    model.eval()
    print(f"  ✅ Loaded: {hf_id}")
    return model, tokenizer


def generate_texts(prompts, model, tokenizer,
                   prompt_style, batch_size=8):
    """
    Generate responses for a list of raw prompt strings.
    Processes in batches to avoid OOM.
    Returns a list of generated strings (input prompt stripped).
    """
    generated = []

    for i in tqdm(range(0, len(prompts), batch_size),
                  desc="Generating"):
        batch_raw = prompts[i : i + batch_size]

        # Format each prompt correctly for this model family
        batch_formatted = [
            format_prompt(p, prompt_style, tokenizer)
            for p in batch_raw
        ]

        inputs = tokenizer(
            batch_formatted,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512,
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=True,
                temperature=0.7,
                top_p=0.9,
                repetition_penalty=1.1,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )

        # Strip input tokens — keep only the newly generated part
        input_len = inputs["input_ids"].shape[1]
        for out in outputs:
            new_tokens = out[input_len:]
            text = tokenizer.decode(
                new_tokens,
                skip_special_tokens=True,
            ).strip()
            generated.append(text)

    return generated


print("✅ format_prompt, load_causal_model, generate_texts defined")

# ================================================================
# CELL S4: LOAD TEST DATA & BUILD PROMPT POOL
# ================================================================
hc3_test  = pd.read_csv("hc3_test.csv")
eli5_test = pd.read_csv("eli5_test.csv")

# Questions only from human rows — used as generation prompts
hc3_prompts = (hc3_test[hc3_test["label"] == "human"]["text"]
               .sample(N_SAMPLES, random_state=SEED)
               .reset_index(drop=True))

eli5_prompts = (eli5_test[eli5_test["label"] == "human"]["text"]
                .sample(N_SAMPLES, random_state=SEED)
                .reset_index(drop=True))

# Human texts for pairing later in evaluation
hc3_human_texts  = hc3_test[hc3_test["label"] == "human"]["text"].tolist()
eli5_human_texts = eli5_test[eli5_test["label"] == "human"]["text"].tolist()

print(f"HC3  prompt pool : {len(hc3_prompts)}")
print(f"ELI5 prompt pool : {len(eli5_prompts)}")
print(f"HC3  human pool  : {len(hc3_human_texts)}")
print(f"ELI5 human pool  : {len(eli5_human_texts)}")
# ================================================================
# CELL G1: GENERATE MULTI-LLM CORPUS
# Runs one model at a time, saves after each to survive crashes.
# ================================================================
generated_corpus = {}

corpus_path = "./stage3_results/generated_corpus.pkl"
if os.path.exists(corpus_path):
    with open(corpus_path, "rb") as f:
        generated_corpus = pickle.load(f)
    print(f"Resumed corpus: {list(generated_corpus.keys())}")

for model_name, model_cfg in MODEL_REGISTRY.items():
    if model_name in generated_corpus:
        print(f"  ✅ {model_name} already generated — skipping")
        continue

    print(f"\n{'='*60}\nGenerating: {model_name}\n{'='*60}")
    try:
        model, tokenizer = load_causal_model(model_cfg)

        hc3_gen = generate_texts(
            hc3_prompts.tolist(),
            model, tokenizer,
            prompt_style=model_cfg["prompt_style"],
        )
        eli5_gen = generate_texts(
            eli5_prompts.tolist(),
            model, tokenizer,
            prompt_style=model_cfg["prompt_style"],
        )

        generated_corpus[model_name] = {
            "hc3":  hc3_gen,
            "eli5": eli5_gen,
        }

        del model, tokenizer
        torch.cuda.empty_cache()

        # Checkpoint after every model — so a crash mid-loop
        # doesn't lose already-generated data
        with open(corpus_path, "wb") as f:
            pickle.dump(generated_corpus, f)
        print(f"  ✅ Saved corpus checkpoint after {model_name}")

    except Exception as e:
        print(f"  ❌ {model_name} failed: {e}")

print(f"\n✅ Corpus complete: {list(generated_corpus.keys())}")

In [ ]:
# ================================================================
# CELL SANITY: Dataset sanity check + JSON export + download
# ================================================================
from google.colab import files
import json, textwrap

# ── Build flat dataset from generated_corpus ─────────────────
dataset_records = []

for model_name, splits in generated_corpus.items():
    for dataset_name, texts in splits.items():   # dataset_name = "hc3" or "eli5"
        for idx, text in enumerate(texts):
            dataset_records.append({
                "id":           f"{model_name}_{dataset_name}_{idx}",
                "source_model": model_name,
                "dataset":      dataset_name,
                "label":        "llm",
                "text":         text,
            })

# Also add human references so the final JSON is self-contained
for idx, text in enumerate(hc3_human_texts):
    dataset_records.append({
        "id":           f"human_hc3_{idx}",
        "source_model": "human",
        "dataset":      "hc3",
        "label":        "human",
        "text":         text,
    })

for idx, text in enumerate(eli5_human_texts):
    dataset_records.append({
        "id":           f"human_eli5_{idx}",
        "source_model": "human",
        "dataset":      "eli5",
        "label":        "human",
        "text":         text,
    })

dataset_df = pd.DataFrame(dataset_records)

# ── Sanity checks ─────────────────────────────────────────────
print("=" * 60)
print("DATASET SANITY CHECK")
print("=" * 60)

print(f"\nTotal records      : {len(dataset_df):,}")
print(f"Columns            : {list(dataset_df.columns)}")
print(f"Null values        :\n{dataset_df.isnull().sum()}")

print("\n── Label distribution ───────────────────────────────────")
print(dataset_df.groupby(["source_model", "dataset", "label"])
      .size()
      .reset_index(name="count")
      .to_string(index=False))

print("\n── Text length stats (chars) ────────────────────────────")
dataset_df["text_len"] = dataset_df["text"].str.len()
print(dataset_df.groupby(["source_model"])["text_len"]
      .agg(["mean", "min", "max"])
      .round(1)
      .to_string())

print("\n── Sample records (2 per source model) ──────────────────")
for model_name in dataset_df["source_model"].unique():
    sample = dataset_df[dataset_df["source_model"] == model_name].head(2)
    for _, row in sample.iterrows():
        print(f"\n  [{row['source_model']} | {row['dataset']} | {row['label']}]")
        # Wrap text so it doesn't flood the output
        wrapped = textwrap.fill(str(row["text"])[:300], width=80,
                                initial_indent="  ", subsequent_indent="  ")
        print(wrapped)
        if len(row["text"]) > 300:
            print("  ...")

# ── Check for empty texts ─────────────────────────────────────
empty = dataset_df[dataset_df["text"].str.strip() == ""]
if len(empty) > 0:
    print(f"\n⚠️  WARNING: {len(empty)} empty text entries found!")
    print(empty[["id", "source_model", "dataset"]].to_string(index=False))
else:
    print(f"\n✅ No empty texts found")

# ── Check for duplicates ──────────────────────────────────────
dupes = dataset_df["text"].duplicated().sum()
if dupes > 0:
    print(f"⚠️  WARNING: {dupes} duplicate texts found")
else:
    print(f"✅ No duplicate texts found")

print("\n✅ Sanity check complete")

# ── Save JSON ─────────────────────────────────────────────────
json_path = "./stage3_results/cross_llm_evaluation_dataset.json"

dataset_df.drop(columns=["text_len"], inplace=True)   # drop helper col

with open(json_path, "w", encoding="utf-8") as f:
    json.dump(dataset_records, f, indent=2, ensure_ascii=False)

print(f"\n✅ Saved JSON: {json_path}")
print(f"   File size : {os.path.getsize(json_path) / 1024 / 1024:.2f} MB")

# ── Download ──────────────────────────────────────────────────
files.download(json_path)
print("📥 Download triggered")

In [ ]:
# ================================================================
# CORE PYTHON
# ================================================================
import os
import json
import glob
import pickle
import random

# ================================================================
# NUMERICAL / DATA
# ================================================================
import numpy as np
import pandas as pd

# ================================================================
# PYTORCH
# ================================================================
import torch
from torch.utils.data import Dataset, DataLoader

# ================================================================
# HUGGINGFACE
# ================================================================
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

from huggingface_hub import (
    snapshot_download,
    login
)

# ================================================================
# COLAB UTILITIES
# ================================================================
from google.colab import userdata

# ================================================================
# MACHINE LEARNING / METRICS
# ================================================================
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    roc_curve
)

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# ================================================================
# STATISTICS
# ================================================================
import scipy.stats as scipy_stats

# ================================================================
# VISUALIZATION
# ================================================================
import matplotlib.pyplot as plt
import seaborn as sns

# ================================================================
# PROGRESS
# ================================================================
from tqdm import tqdm

In [ ]:
# ================================================================
# DEVICE SETUP
# ================================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

In [ ]:
# ================================================================
# CELL LOAD TRAINED NEURAL DETECTORS
# (Local-first, HF Hub fallback)
# ================================================================
import json
import glob as glob_mod
from huggingface_hub import snapshot_download
from google.colab import userdata
from huggingface_hub import login
import numpy as np


import os, json, glob as glob_mod
from torch.utils.data import Dataset, DataLoader   # ← add this
from huggingface_hub import snapshot_download, login
from google.colab import userdata

HF_TOKEN  = userdata.get("HF_TOKEN")
HF_USER   = "Moodlerz"
login(token=HF_TOKEN, add_to_git_credential=False)

# ================================================================
# CELL: LOAD HUMAN TEXT POOLS — Run on every fresh runtime
# ================================================================
import pandas as pd

hc3_test  = pd.read_csv("hc3_test.csv")
eli5_test = pd.read_csv("eli5_test.csv")

hc3_human_texts  = hc3_test[hc3_test["label"] == "human"]["text"].tolist()
eli5_human_texts = eli5_test[eli5_test["label"] == "human"]["text"].tolist()

print(f"HC3  human pool : {len(hc3_human_texts)}")
print(f"ELI5 human pool : {len(eli5_human_texts)}")


# Maps detector name → (local_dir, hf_repo_suffix)
DETECTOR_REGISTRY = {
    "BERT-HC3":       ("./models/BERT_hc3",       "bert-detector-hc3"),
    "RoBERTa-HC3":    ("./models/RoBERTa_hc3",    "roberta-detector-hc3"),
    "ELECTRA-HC3":    ("./models/ELECTRA_hc3",     "electra-detector-hc3"),
    "DistilBERT-HC3": ("./models/DistilBERT_hc3",  "distilbert-detector-hc3"),
    "DeBERTa-HC3":    ("./models/DeBERTa_hc3",     "deberta-v3-detector-hc3"),
}


def find_best_checkpoint(model_dir):
    """
    Priority 1: Read trainer_state.json → best_model_checkpoint by ROC-AUC
    Priority 2: No trainer_state → take highest-numbered checkpoint
    Priority 3: No checkpoint subdirs → return root dir (DeBERTa case)
    """
    ckpts = glob_mod.glob(os.path.join(model_dir, "checkpoint-*"))

    if not ckpts:
        # DeBERTa — saved directly to root, no checkpoint subdirs
        print(f"      No checkpoint subdirs — using root dir: {model_dir}")
        return model_dir

    # Try to read best_model_checkpoint from trainer_state.json
    for ckpt in ckpts:
        state_file = os.path.join(ckpt, "trainer_state.json")
        if os.path.exists(state_file):
            with open(state_file, "r") as f:
                state = json.load(f)
            best = state.get("best_model_checkpoint")
            if best and os.path.exists(best):
                print(f"      ✅ Best checkpoint (by ROC-AUC): {best}")
                return best
            else:
                print(f"      ⚠️  trainer_state found but best_model_checkpoint "
                      f"missing or invalid: {best}")

    # Fallback — no valid trainer_state, use highest-numbered
    latest = max(ckpts, key=lambda x: int(x.split("-")[-1]))
    print(f"      ⚠️  No trainer_state found — falling back to latest: {latest}")
    return latest


def resolve_detector_path(det_name, local_dir, hf_repo_suffix):
    """
    1. Check if local_dir exists and has model weights → use it.
    2. Otherwise pull from HF Hub into local_dir and use that.
    Returns the resolved checkpoint path ready for from_pretrained.
    """
    model_files = ["pytorch_model.bin", "model.safetensors"]

    # ── Step 1: Check local ───────────────────────────────────
    print(f"\n  [{det_name}] Checking local: {local_dir}")

    local_has_model = False
    if os.path.exists(local_dir):
        # Check root dir
        for mf in model_files:
            if os.path.exists(os.path.join(local_dir, mf)):
                local_has_model = True
                break
        # Check checkpoint subdirs
        if not local_has_model:
            ckpts = glob_mod.glob(os.path.join(local_dir, "checkpoint-*"))
            for ckpt in ckpts:
                for mf in model_files:
                    if os.path.exists(os.path.join(ckpt, mf)):
                        local_has_model = True
                        break

    if local_has_model:
        print(f"  ✅ Found locally — using local weights")
        resolved = find_best_checkpoint(local_dir)
        return resolved

    # ── Step 2: Pull from HF Hub ──────────────────────────────
    repo_id = f"{HF_USER}/{hf_repo_suffix}"
    print(f"  ⚠️  Not found locally")
    print(f"  🔄 Pulling from HF Hub: {repo_id}")

    try:
        snapshot_download(
            repo_id=repo_id,
            local_dir=local_dir,
            token=HF_TOKEN,
            ignore_patterns=["*.msgpack", "flax_model*", "tf_model*"],
        )
        print(f"  ✅ Downloaded from HF Hub → {local_dir}")
        resolved = find_best_checkpoint(local_dir)
        return resolved

    except Exception as e:
        print(f"  ❌ HF Hub pull failed for {repo_id}: {str(e)}")
        return None


# ── Resolve all detectors ─────────────────────────────────────
NEURAL_DETECTORS = {}   # {det_name: (resolved_ckpt_path, base_model)}

BASE_MODELS = {
    "BERT-HC3":       "bert-base-uncased",
    "RoBERTa-HC3":    "roberta-base",
    "ELECTRA-HC3":    "google/electra-base-discriminator",
    "DistilBERT-HC3": "distilbert-base-uncased",
    "DeBERTa-HC3":    "microsoft/deberta-v3-base",
}

print("=" * 60)
print("RESOLVING DETECTOR CHECKPOINTS")
print("=" * 60)

for det_name, (local_dir, hf_suffix) in DETECTOR_REGISTRY.items():
    resolved = resolve_detector_path(det_name, local_dir, hf_suffix)
    if resolved is not None:
        NEURAL_DETECTORS[det_name] = (resolved, BASE_MODELS[det_name])
        print(f"  ✅ {det_name:20s} ready → {resolved}")
    else:
        print(f"  ❌ {det_name:20s} UNAVAILABLE — will be skipped")

print(f"\nResolved {len(NEURAL_DETECTORS)}/{len(DETECTOR_REGISTRY)} detectors")
print("=" * 60)

In [ ]:

# ================================================================
# CELL: DETECTOR INFERENCE ENGINE
# ================================================================
class InferenceDataset(Dataset):
    def __init__(self, texts, tokenizer, max_len=512):
        self.texts = texts
        self.tok   = tokenizer
        self.max_len = max_len
    def __len__(self): return len(self.texts)
    def __getitem__(self, i):
        enc = self.tok(str(self.texts[i]), max_length=self.max_len,
                       padding="max_length", truncation=True,
                       return_tensors="pt")
        return {"input_ids":      enc["input_ids"].flatten(),
                "attention_mask": enc["attention_mask"].flatten()}

def run_detector(texts, ckpt_path, base_model):
    tokenizer = AutoTokenizer.from_pretrained(ckpt_path)
    model     = AutoModelForSequenceClassification.from_pretrained(
                    ckpt_path).to(device)
    model.eval()
    ds     = InferenceDataset(texts, tokenizer)
    loader = DataLoader(ds, batch_size=32, shuffle=False, num_workers=2)
    scores = []
    with torch.no_grad():
        for batch in loader:
            logits = model(input_ids=batch["input_ids"].to(device),
                           attention_mask=batch["attention_mask"].to(device)).logits
            scores.extend(torch.softmax(logits, dim=-1)[:,1].cpu().numpy())
    del model, tokenizer; torch.cuda.empty_cache()
    return np.array(scores)

# ================================================================
# CELL: five_metrics, bootstrap_five, delong_p
# ================================================================
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    brier_score_loss, roc_curve,
)

def five_metrics(y_true, y_score):
    """
    Compute the 5 core metrics used throughout Stage 3:
    AUROC, AUPRC, EER, Brier Score, FPR@95TPR
    """
    auroc = roc_auc_score(y_true, y_score)
    auprc = average_precision_score(y_true, y_score)
    brier = brier_score_loss(y_true, y_score)

    # EER — point where FPR ≈ FNR (1 - TPR)
    fpr, tpr, _ = roc_curve(y_true, y_score)
    fnr = 1 - tpr
    eer_idx = np.argmin(np.abs(fpr - fnr))
    eer = float((fpr[eer_idx] + fnr[eer_idx]) / 2)

    # FPR @ 95% TPR — false positive rate when TPR = 0.95
    tpr_95_idx = np.argmin(np.abs(tpr - 0.95))
    fpr95 = float(fpr[tpr_95_idx])

    return {
        "auroc": auroc,
        "auprc": auprc,
        "eer":   eer,
        "brier": brier,
        "fpr95": fpr95,
    }


def bootstrap_five(y_true, y_score, n_boot=1000, seed=42):
    """
    Bootstrap 95% CIs for all five metrics.
    Returns dict of {metric: (lower, upper)}
    """
    rng    = np.random.default_rng(seed)
    n      = len(y_true)
    keys   = ["auroc", "auprc", "eer", "brier", "fpr95"]
    boots  = {k: [] for k in keys}

    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        yt  = y_true[idx]
        ys  = y_score[idx]

        # Skip degenerate samples (only one class present)
        if len(np.unique(yt)) < 2:
            continue

        m = five_metrics(yt, ys)
        for k in keys:
            boots[k].append(m[k])

    ci = {}
    for k in keys:
        arr = np.array(boots[k])
        ci[k] = (
            float(np.percentile(arr, 2.5)),
            float(np.percentile(arr, 97.5)),
        )
    return ci


def delong_p(y_true, y_score_a, y_score_b):
    """
    DeLong test for comparing two AUROCs on the same test set.
    Returns: (auc_a, auc_b, z_stat, p_value)

    Based on: DeLong et al. (1988) — Comparing the Areas Under
    Two or More Correlated Receiver Operating Characteristic Curves.
    """
    def auc_and_structural_components(y_true, y_score):
        pos = y_score[y_true == 1]
        neg = y_score[y_true == 0]
        n_pos, n_neg = len(pos), len(neg)

        # Placement values
        v10 = np.array([
            (np.sum(p > neg) + 0.5 * np.sum(p == neg)) / n_neg
            for p in pos
        ])
        v01 = np.array([
            (np.sum(n < pos) + 0.5 * np.sum(n == pos)) / n_pos
            for n in neg
        ])

        auc = v10.mean()
        return auc, v10, v01, n_pos, n_neg

    auc_a, v10_a, v01_a, n_pos, n_neg = auc_and_structural_components(
        y_true, y_score_a)
    auc_b, v10_b, v01_b, _,     _     = auc_and_structural_components(
        y_true, y_score_b)

    # Variance components
    s10 = np.cov(v10_a, v10_b)   # 2x2 covariance matrix
    s01 = np.cov(v01_a, v01_b)

    var_a  = s10[0, 0] / n_pos + s01[0, 0] / n_neg
    var_b  = s10[1, 1] / n_pos + s01[1, 1] / n_neg
    cov_ab = s10[0, 1] / n_pos + s01[0, 1] / n_neg

    var_diff = var_a + var_b - 2 * cov_ab

    if var_diff <= 0:
        # Degenerate case — return p=1 (no detectable difference)
        return auc_a, auc_b, 0.0, 1.0

    z = (auc_a - auc_b) / np.sqrt(var_diff)
    p = 2 * (1 - scipy_stats.norm.cdf(abs(z)))   # two-tailed

    return auc_a, auc_b, float(z), float(p)


print("✅ five_metrics, bootstrap_five, delong_p defined")

results_3A = {}   # {detector: {llm_name: {dataset: metrics+ci}}}

for det_name, (ckpt, base) in NEURAL_DETECTORS.items():
    if not os.path.exists(ckpt):
        print(f"⚠️  Skipping {det_name} — checkpoint not found")
        continue
    print(f"\n{'='*60}\n3A | Detector: {det_name}\n{'='*60}")
    results_3A[det_name] = {}

    for llm_name, corpus in generated_corpus.items():
        results_3A[det_name][llm_name] = {}

        for ds_name, human_pool in [("hc3",  hc3_human_texts),
                                     ("eli5", eli5_human_texts)]:
            llm_texts   = corpus[ds_name]
            human_texts = human_pool[:len(llm_texts)]

            all_texts = human_texts + llm_texts
            y_true    = np.array([0]*len(human_texts) + [1]*len(llm_texts))

            print(f"  [{llm_name}][{ds_name}] scoring {len(all_texts)} texts ...")
            scores = run_detector(all_texts, ckpt, base)

            m  = five_metrics(y_true, scores)
            ci = bootstrap_five(y_true, scores)
            results_3A[det_name][llm_name][ds_name] = {
                "metrics": m, "ci": ci,
                "y_true": y_true, "y_score": scores,
            }
            print(f"    AUROC={m['auroc']:.3f}  AUPRC={m['auprc']:.3f}"
                  f"  EER={m['eer']:.3f}  Brier={m['brier']:.3f}"
                  f"  FPR@95={m['fpr95']:.3f}")

with open("./stage3_results/3A/results_3A.pkl","wb") as f:
    pickle.dump(results_3A, f)
print("\n✅ Stage 3A complete — results saved")

# ================================================================
# CELL: VISUALISE — Detector × Source-LLM AUROC Matrix
# ================================================================
for ds_name in ["hc3","eli5"]:
    det_names = list(results_3A.keys())
    llm_names = list(generated_corpus.keys())
    mat = np.full((len(det_names), len(llm_names)), np.nan)

    for i, det in enumerate(det_names):
        for j, llm in enumerate(llm_names):
            try:
                mat[i,j] = results_3A[det][llm][ds_name]["metrics"]["auroc"]
            except: pass

    fig, ax = plt.subplots(figsize=(max(8,len(llm_names)*1.4),
                                    max(4,len(det_names)*0.8)))
    sns.heatmap(mat, annot=True, fmt=".3f", cmap="RdYlGn",
                vmin=0.5, vmax=1.0,
                xticklabels=llm_names, yticklabels=det_names,
                linewidths=0.5, ax=ax)
    ax.set_title(f"3A: Detector × Source-LLM AUROC Matrix [{ds_name.upper()}]\n"
                 f"Zero-shot transfer — no retraining",
                 fontsize=13, fontweight="bold")
    ax.set_xlabel("Source LLM (test)"); ax.set_ylabel("Detector (trained on HC3)")
    plt.tight_layout()
    plt.savefig(f"./stage3_results/3A/auroc_matrix_{ds_name}.png",
                dpi=150, bbox_inches="tight")
    plt.show()

# ================================================================
# CELL: FULL 5-METRIC TABLE WITH CIs
# ================================================================
rows = []
for det in results_3A:
    for llm in results_3A[det]:
        for ds in results_3A[det][llm]:
            m  = results_3A[det][llm][ds]["metrics"]
            ci = results_3A[det][llm][ds]["ci"]
            rows.append({
                "Detector": det, "Source_LLM": llm, "Dataset": ds,
                "AUROC":  m["auroc"],
                "AUROC_CI": f"[{ci['auroc'][0]:.3f},{ci['auroc'][1]:.3f}]",
                "AUPRC":  m["auprc"],
                "EER":    m["eer"],
                "Brier":  m["brier"],
                "FPR@95": m["fpr95"],
            })

df_3A = pd.DataFrame(rows)
print(df_3A.round(3).to_string(index=False))
df_3A.to_csv("./stage3_results/3A/table_3A.csv", index=False)
print("\n✅ Saved: table_3A.csv")

# ================================================================
# CELL: DELONG PAIRWISE TEST (per LLM, per dataset)
# ================================================================
for llm_name in generated_corpus:
    for ds_name in ["hc3","eli5"]:
        valid_dets = [d for d in results_3A
                      if llm_name in results_3A[d]
                      and ds_name in results_3A[d][llm_name]]
        if len(valid_dets) < 2: continue

        # All must share same y_true
        ref_yt = results_3A[valid_dets[0]][llm_name][ds_name]["y_true"]

        n = len(valid_dets)
        pmat = np.ones((n,n))
        for i in range(n):
            for j in range(i+1,n):
                sa = results_3A[valid_dets[i]][llm_name][ds_name]["y_score"]
                sb = results_3A[valid_dets[j]][llm_name][ds_name]["y_score"]
                if not np.array_equal(
                        results_3A[valid_dets[i]][llm_name][ds_name]["y_true"],
                        results_3A[valid_dets[j]][llm_name][ds_name]["y_true"]):
                    continue
                _, p, _, _ = delong_p(ref_yt, sa, sb)
                pmat[i,j] = pmat[j,i] = p

        annot = np.where(pmat<0.001,"***",
                np.where(pmat<0.01,"**",
                np.where(pmat<0.05,"*","ns")))
        fig, ax = plt.subplots(figsize=(6,5))
        sns.heatmap(pd.DataFrame(pmat, index=valid_dets, columns=valid_dets),
                    annot=annot, fmt="", cmap="RdYlGn_r",
                    vmin=0, vmax=0.1, linewidths=0.5, ax=ax,
                    mask=np.eye(n,dtype=bool))
        ax.set_title(f"DeLong test — {llm_name} / {ds_name.upper()}",
                     fontsize=11, fontweight="bold")
        plt.tight_layout()
        plt.savefig(f"./stage3_results/3A/delong_{llm_name}_{ds_name}.png",
                    dpi=150, bbox_inches="tight")
        plt.show()

In [ ]:
# ================================================================
# CELL: EMBED ALL LLM TEXTS (sentence-transformers)
# ================================================================
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sentence_transformers import SentenceTransformer

N_SAMPLES = 200
SEED      = 42

print("Loading sentence-transformer for 3B embeddings ...")
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

embeddings_3B = {}


from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    brier_score_loss, roc_curve,
)

def five_metrics(y_true, y_score):
    """
    Compute the 5 core metrics used throughout Stage 3:
    AUROC, AUPRC, EER, Brier Score, FPR@95TPR
    """
    auroc = roc_auc_score(y_true, y_score)
    auprc = average_precision_score(y_true, y_score)
    brier = brier_score_loss(y_true, y_score)

    # EER — point where FPR ≈ FNR (1 - TPR)
    fpr, tpr, _ = roc_curve(y_true, y_score)
    fnr = 1 - tpr
    eer_idx = np.argmin(np.abs(fpr - fnr))
    eer = float((fpr[eer_idx] + fnr[eer_idx]) / 2)

    # FPR @ 95% TPR — false positive rate when TPR = 0.95
    tpr_95_idx = np.argmin(np.abs(tpr - 0.95))
    fpr95 = float(fpr[tpr_95_idx])

    return {
        "auroc": auroc,
        "auprc": auprc,
        "eer":   eer,
        "brier": brier,
        "fpr95": fpr95,
    }


def bootstrap_five(y_true, y_score, n_boot=1000, seed=42):
    """
    Bootstrap 95% CIs for all five metrics.
    Returns dict of {metric: (lower, upper)}
    """
    rng    = np.random.default_rng(seed)
    n      = len(y_true)
    keys   = ["auroc", "auprc", "eer", "brier", "fpr95"]
    boots  = {k: [] for k in keys}

    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        yt  = y_true[idx]
        ys  = y_score[idx]

        # Skip degenerate samples (only one class present)
        if len(np.unique(yt)) < 2:
            continue

        m = five_metrics(yt, ys)
        for k in keys:
            boots[k].append(m[k])

    ci = {}
    for k in keys:
        arr = np.array(boots[k])
        ci[k] = (
            float(np.percentile(arr, 2.5)),
            float(np.percentile(arr, 97.5)),
        )
    return ci


def delong_p(y_true, y_score_a, y_score_b):
    """
    DeLong test for comparing two AUROCs on the same test set.
    Returns: (auc_a, auc_b, z_stat, p_value)

    Based on: DeLong et al. (1988) — Comparing the Areas Under
    Two or More Correlated Receiver Operating Characteristic Curves.
    """
    def auc_and_structural_components(y_true, y_score):
        pos = y_score[y_true == 1]
        neg = y_score[y_true == 0]
        n_pos, n_neg = len(pos), len(neg)

        # Placement values
        v10 = np.array([
            (np.sum(p > neg) + 0.5 * np.sum(p == neg)) / n_neg
            for p in pos
        ])
        v01 = np.array([
            (np.sum(n < pos) + 0.5 * np.sum(n == pos)) / n_pos
            for n in neg
        ])

        auc = v10.mean()
        return auc, v10, v01, n_pos, n_neg

    auc_a, v10_a, v01_a, n_pos, n_neg = auc_and_structural_components(
        y_true, y_score_a)
    auc_b, v10_b, v01_b, _,     _     = auc_and_structural_components(
        y_true, y_score_b)

    # Variance components
    s10 = np.cov(v10_a, v10_b)   # 2x2 covariance matrix
    s01 = np.cov(v01_a, v01_b)

    var_a  = s10[0, 0] / n_pos + s01[0, 0] / n_neg
    var_b  = s10[1, 1] / n_pos + s01[1, 1] / n_neg
    cov_ab = s10[0, 1] / n_pos + s01[0, 1] / n_neg

    var_diff = var_a + var_b - 2 * cov_ab

    if var_diff <= 0:
        # Degenerate case — return p=1 (no detectable difference)
        return auc_a, auc_b, 0.0, 1.0

    z = (auc_a - auc_b) / np.sqrt(var_diff)
    p = 2 * (1 - scipy_stats.norm.cdf(abs(z)))   # two-tailed

    return auc_a, auc_b, float(z), float(p)


print("✅ five_metrics, bootstrap_five, delong_p defined")


for llm_name, corpus in generated_corpus.items():
    embeddings_3B[llm_name] = {}
    for ds_name in ["hc3", "eli5"]:
        texts = corpus[ds_name]
        embs  = embedder.encode(texts, batch_size=64,
                                show_progress_bar=True,
                                convert_to_numpy=True)
        embeddings_3B[llm_name][ds_name] = embs
        print(f"  {llm_name}/{ds_name}: {embs.shape}")


HUMAN_TRAIN_N = N_SAMPLES   # indices   0 : 200
HUMAN_TEST_N  = N_SAMPLES   # indices 200 : 400
HUMAN_TOTAL   = HUMAN_TRAIN_N + HUMAN_TEST_N

assert len(hc3_human_texts)  >= HUMAN_TOTAL, \
    f"HC3 human pool too small ({len(hc3_human_texts)} < {HUMAN_TOTAL})"
assert len(eli5_human_texts) >= HUMAN_TOTAL, \
    f"ELI5 human pool too small ({len(eli5_human_texts)} < {HUMAN_TOTAL})"

print(f"\nEmbedding {HUMAN_TOTAL} human texts per dataset ...")

_hc3_all  = embedder.encode(hc3_human_texts[:HUMAN_TOTAL],  batch_size=64,
                             show_progress_bar=True, convert_to_numpy=True)
_eli5_all = embedder.encode(eli5_human_texts[:HUMAN_TOTAL], batch_size=64,
                             show_progress_bar=True, convert_to_numpy=True)

human_embs_hc3_train  = _hc3_all[:HUMAN_TRAIN_N]
human_embs_hc3_test   = _hc3_all[HUMAN_TRAIN_N:]
human_embs_eli5_train = _eli5_all[:HUMAN_TRAIN_N]
human_embs_eli5_test  = _eli5_all[HUMAN_TRAIN_N:]

print(f"  HC3  train={human_embs_hc3_train.shape}  test={human_embs_hc3_test.shape}")
print(f"  ELI5 train={human_embs_eli5_train.shape}  test={human_embs_eli5_test.shape}")

with open("./stage3_results/3B/embeddings_3B.pkl", "wb") as f:
    pickle.dump({
        "llm":              embeddings_3B,
        "human_hc3_train":  human_embs_hc3_train,
        "human_hc3_test":   human_embs_hc3_test,
        "human_eli5_train": human_embs_eli5_train,
        "human_eli5_test":  human_embs_eli5_test,
    }, f)
print("✅ Embeddings saved")


# ================================================================
# CELL 3B-2: TRAIN × TEST AUROC MATRIX
# ================================================================
llm_names = list(generated_corpus.keys())
results_3B = {}

CLASSIFIERS = {
    "LogisticRegression": lambda: LogisticRegression(
        max_iter=1000, C=1.0, random_state=SEED),
    "SVM": lambda: SVC(
        kernel="rbf", C=1.0, probability=True, random_state=SEED),
    "RandomForest": lambda: RandomForestClassifier(
        n_estimators=200, max_depth=None, random_state=SEED, n_jobs=-1),
}

for clf_name in CLASSIFIERS:
    results_3B[clf_name] = {}

# Each dataset gets its own disjoint human train/test split
dataset_splits = [
    ("hc3",  human_embs_hc3_train,  human_embs_hc3_test),
    ("eli5", human_embs_eli5_train, human_embs_eli5_test),
]

for ds_name, h_train, h_test in dataset_splits:
    print(f"\n{'='*60}\n3B Matrix — {ds_name.upper()}\n{'='*60}")

    for train_llm in llm_names:
        # Training: LLM texts (1) + TRAIN-ONLY human texts (0)
        X_train = np.vstack([embeddings_3B[train_llm][ds_name], h_train])
        y_train = np.array([1]*N_SAMPLES + [0]*len(h_train))

        # Scaler fitted only on training data
        scaler = StandardScaler()
        X_tr   = scaler.fit_transform(X_train)

        for clf_name, clf_factory in CLASSIFIERS.items():
            results_3B[clf_name].setdefault(train_llm, {})

            print(f"\n  Classifier: {clf_name}  |  Train LLM: {train_llm}")
            clf = clf_factory()
            clf.fit(X_tr, y_train)

            for test_llm in llm_names:
                # Testing: LLM texts (1) + TEST-ONLY human texts (0)
                # h_test was NEVER seen during training → no leakage
                X_test = np.vstack([embeddings_3B[test_llm][ds_name], h_test])
                y_test = np.array([1]*N_SAMPLES + [0]*len(h_test))
                X_te   = scaler.transform(X_test)
                scores = clf.predict_proba(X_te)[:, 1]

                m  = five_metrics(y_test, scores)
                ci = bootstrap_five(y_test, scores)
                results_3B[clf_name][train_llm].setdefault(test_llm, {})[ds_name] = {
                    "metrics": m, "ci": ci,
                    "y_true":  y_test,
                    "y_score": scores,
                }
                print(f"    Train={train_llm:15s}  Test={test_llm:15s}"
                      f"  AUROC={m['auroc']:.3f}")

with open("./stage3_results/3B/results_3B.pkl", "wb") as f:
    pickle.dump(results_3B, f)
print("\n✅ Stage 3B complete")


# ================================================================
# CELL:VISUALISE — Train × Test AUROC HEATMAP
# ================================================================
for clf_name in CLASSIFIERS:
    for ds_name in ["hc3", "eli5"]:
        n   = len(llm_names)
        mat = np.full((n, n), np.nan)

        for i, tr in enumerate(llm_names):
            for j, te in enumerate(llm_names):
                try:
                    mat[i, j] = results_3B[clf_name][tr][te][ds_name]["metrics"]["auroc"]
                except:
                    pass

        fig, ax = plt.subplots(figsize=(max(8, n*1.2), max(6, n*1.0)))
        sns.heatmap(mat, annot=True, fmt=".3f", cmap="RdYlGn",
                    vmin=0.5, vmax=1.0,
                    xticklabels=llm_names, yticklabels=llm_names,
                    linewidths=0.5, ax=ax,
                    cbar_kws={"label": "AUROC"})
        for k in range(n):
            ax.add_patch(plt.Rectangle((k, k), 1, 1, fill=False,
                                       edgecolor="black", lw=2.5))
        ax.set_title(f"3B [{clf_name}]: Train × Test AUROC — {ds_name.upper()}\n"
                     f"Diagonal = in-distribution  |  Off-diagonal = cross-LLM\n"
                     f"(Leakage-free: disjoint human train/test splits)",
                     fontsize=12, fontweight="bold")
        ax.set_xlabel("Test LLM")
        ax.set_ylabel("Train LLM")
        plt.tight_layout()
        plt.savefig(f"./stage3_results/3B/matrix_{clf_name}_{ds_name}.png",
                    dpi=150, bbox_inches="tight")
        plt.show()


# ================================================================
# CELL: TRANSFER DEGRADATION — diagonal vs off-diagonal
# ================================================================
for clf_name in CLASSIFIERS:
    for ds_name in ["hc3", "eli5"]:
        in_dist, cross = [], []

        for i, tr in enumerate(llm_names):
            for j, te in enumerate(llm_names):
                try:
                    auc = results_3B[clf_name][tr][te][ds_name]["metrics"]["auroc"]
                    if i == j:
                        in_dist.append(auc)
                    else:
                        cross.append(auc)
                except:
                    pass

        fig, ax = plt.subplots(figsize=(7, 5))
        ax.boxplot([in_dist, cross],
                   labels=["In-distribution", "Cross-LLM"],
                   patch_artist=True,
                   boxprops=dict(facecolor="#3498db", alpha=0.6))
        ax.set_ylabel("AUROC")
        ax.grid(axis="y", alpha=0.3)

        t_stat, p_val = scipy_stats.ttest_ind(in_dist, cross)
        ax.set_title(f"3B [{clf_name}]: In-dist vs Cross-LLM — {ds_name.upper()}\n"
                     f"t={t_stat:.2f}  p={p_val:.4f}",
                     fontsize=12)
        plt.tight_layout()
        plt.savefig(f"./stage3_results/3B/transfer_degradation_{clf_name}_{ds_name}.png",
                    dpi=150)
        plt.show()

        print(f"[{clf_name}][{ds_name}]  "
              f"In-dist mean={np.mean(in_dist):.3f}  "
              f"Cross mean={np.mean(cross):.3f}  "
              f"Δ={np.mean(in_dist)-np.mean(cross):.3f}")


# ================================================================
# CELL: CLASSIFIER COMPARISON BAR CHART
# ================================================================
for ds_name in ["hc3", "eli5"]:
    fig, ax = plt.subplots(figsize=(10, 5))
    x      = np.arange(len(llm_names))
    width  = 0.8 / len(CLASSIFIERS)
    colors = ["#3498db", "#e74c3c", "#2ecc71"]

    for i, clf_name in enumerate(CLASSIFIERS):
        avg_aucs = []
        for tr in llm_names:
            aucs = []
            for te in llm_names:
                try:
                    aucs.append(
                        results_3B[clf_name][tr][te][ds_name]["metrics"]["auroc"])
                except:
                    pass
            avg_aucs.append(np.mean(aucs) if aucs else np.nan)

        ax.bar(x + i*width, avg_aucs, width,
               label=clf_name, color=colors[i], alpha=0.85)

    ax.set_xticks(x + width * len(CLASSIFIERS) / 2)
    ax.set_xticklabels(llm_names, rotation=30, ha="right")
    ax.set_ylabel("Mean AUROC (avg over test LLMs)")
    ax.set_ylim(0.4, 1.05)
    ax.axhline(0.5, color="k", linestyle="--", alpha=0.3, label="Chance")
    ax.set_title(f"3B: LR vs SVM vs RF — Mean AUROC per Train LLM [{ds_name.upper()}]\n"
                 f"(Leakage-free evaluation)",
                 fontsize=12, fontweight="bold")
    ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=9)
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"./stage3_results/3B/classifier_comparison_{ds_name}.png",
                dpi=150, bbox_inches="tight")
    plt.show()


# ================================================================
# CELL: FULL 5-METRIC TABLE WITH CIs
# ================================================================
rows = []
for clf in results_3B:
    for tr in results_3B[clf]:
        for te in results_3B[clf][tr]:
            for ds in results_3B[clf][tr][te]:
                m  = results_3B[clf][tr][te][ds]["metrics"]
                ci = results_3B[clf][tr][te][ds]["ci"]
                rows.append({
                    "Classifier": clf, "Train_LLM": tr,
                    "Test_LLM": te,    "Dataset": ds,
                    "AUROC":  round(m["auroc"], 3),
                    "AUROC_CI": f"[{ci['auroc'][0]:.3f},{ci['auroc'][1]:.3f}]",
                    "AUPRC":  round(m["auprc"], 3),
                    "EER":    round(m["eer"],   3),
                    "Brier":  round(m["brier"], 3),
                    "FPR@95": round(m["fpr95"], 3),
                })

df_3B = pd.DataFrame(rows)
print(df_3B.to_string(index=False))
df_3B.to_csv("./stage3_results/3B/table_3B_corrected.csv", index=False)
print("\n✅ Saved: table_3B_corrected.csv")

In [ ]:
!pip install -q scipy

In [ ]:
from scipy.linalg import sqrtm

# ================================================================
# CELL: EMBED WITH DeBERTa PENULTIMATE LAYER
# ================================================================
N_SAMPLES = 200
SEED      = 42
print("Loading DeBERTa-v3-base for distribution shift embeddings ...")

deberta_ckpt = find_best_checkpoint("./models/DeBERTa_hc3")
deberta_tok  = AutoTokenizer.from_pretrained(deberta_ckpt)
deberta_base = AutoModelForSequenceClassification.from_pretrained(
                   deberta_ckpt, output_hidden_states=True).to(device)
deberta_base.eval()

def embed_deberta(texts, batch_size=32):
    """Extract penultimate hidden state [CLS] from DeBERTa."""
    all_embs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        enc   = deberta_tok(batch, max_length=512, padding=True,
                            truncation=True, return_tensors="pt").to(device)
        with torch.no_grad():
            out = deberta_base(**enc)
        cls_emb = out.hidden_states[-2][:, 0, :].cpu().numpy()
        all_embs.append(cls_emb)
    return np.vstack(all_embs)

# Reference = ChatGPT-generated texts from HC3 test set (what detectors trained on)
hc3_llm_train_texts = hc3_test[hc3_test["label"] == "llm"]["text"].tolist()[:N_SAMPLES]
ref_emb = embed_deberta(hc3_llm_train_texts)
print(f"Reference embedding shape: {ref_emb.shape}")

test_embs_3D = {}
for llm_name, corpus in generated_corpus.items():
    print(f"  Embedding {llm_name} ...")
    test_embs_3D[llm_name] = embed_deberta(corpus["hc3"])

del deberta_base
torch.cuda.empty_cache()
print("✅ DeBERTa embeddings complete")


# ================================================================
# CELL: THREE DISTANCE METRICS
# ================================================================

def kl_gaussian(mu1, cov1, mu2, cov2):
    """KL divergence KL(N1 || N2) — numerically stable via slogdet."""
    d        = len(mu1)
    cov2_inv = np.linalg.pinv(cov2)
    diff     = mu2 - mu1
    term1    = np.trace(cov2_inv @ cov1)
    term2    = float(diff @ cov2_inv @ diff)
    _, logdet1 = np.linalg.slogdet(cov1)
    _, logdet2 = np.linalg.slogdet(cov2)
    term3    = logdet2 - logdet1
    return float(0.5 * (term1 + term2 - d + term3))


def wasserstein2(X, Y):
    """Approximate W2 distance via Gaussian mean + covariance."""
    mu1, mu2 = X.mean(0), Y.mean(0)
    S1, S2   = np.cov(X.T), np.cov(Y.T)
    sq_S2    = sqrtm(S2)
    M        = sqrtm(sq_S2 @ S1 @ sq_S2)
    w2_sq    = np.sum((mu1 - mu2) ** 2) + np.trace(S1 + S2 - 2 * M.real)
    return float(np.sqrt(max(w2_sq, 0)))


def frechet_distance(X, Y):
    """Fréchet distance (FID-style) on embedding space."""
    mu1, mu2 = X.mean(0), Y.mean(0)
    S1, S2   = np.cov(X.T), np.cov(Y.T)
    covmean  = sqrtm(S1 @ S2)
    if np.iscomplexobj(covmean):
        covmean = covmean.real
    fd = np.sum((mu1 - mu2) ** 2) + np.trace(S1 + S2 - 2 * covmean)
    return float(max(fd, 0))


def pca_project(embs, n_comp=64):
    from sklearn.decomposition import PCA
    return PCA(n_components=n_comp, random_state=SEED).fit_transform(embs)


ref_pca = pca_project(ref_emb)

dist_results = {}
for llm_name, emb in test_embs_3D.items():
    te_pca   = pca_project(emb)
    mu1, S1  = ref_pca.mean(0), np.cov(ref_pca.T)
    mu2, S2  = te_pca.mean(0),  np.cov(te_pca.T)

    dist_results[llm_name] = {
        "kl":          kl_gaussian(mu1, S1, mu2, S2),
        "wasserstein": wasserstein2(ref_pca, te_pca),
        "frechet":     frechet_distance(ref_pca, te_pca),
    }
    print(f"  {llm_name:20s}  "
          f"KL={dist_results[llm_name]['kl']:.3f}  "
          f"W2={dist_results[llm_name]['wasserstein']:.3f}  "
          f"FD={dist_results[llm_name]['frechet']:.3f}")


# ================================================================
# CELL: CORRELATE DISTANCE vs AUROC DROP
# ================================================================


IN_DIST_AUROC = {
    "BERT-HC3":       0.9667,
    "RoBERTa-HC3":    0.9977,
    "ELECTRA-HC3":    0.9539,
    "DistilBERT-HC3": 0.9927,
    "DeBERTa-HC3":    0.8770,
}

dist_auroc_rows = []
for det_name in results_3A:
    in_dist_auc = IN_DIST_AUROC.get(det_name, np.nan)

    for llm_name in generated_corpus:
        try:
            cross_auc = results_3A[det_name][llm_name]["hc3"]["metrics"]["auroc"]
            drop      = in_dist_auc - cross_auc
            for dist_type in ["kl", "wasserstein", "frechet"]:
                dist_auroc_rows.append({
                    "Detector":    det_name,
                    "LLM":         llm_name,
                    "dist_type":   dist_type,
                    "distance":    dist_results[llm_name][dist_type],
                    "auroc_drop":  drop,
                    "in_dist_auc": in_dist_auc,
                    "cross_auc":   cross_auc,
                })
        except:
            pass

dist_df = pd.DataFrame(dist_auroc_rows)
os.makedirs("./stage3_results/3D", exist_ok=True)
dist_df.to_csv("./stage3_results/3D/dist_vs_drop.csv", index=False)
print(dist_df.round(4).to_string(index=False))


# ================================================================
# CELL: SPEARMAN CORRELATION & SCATTER PLOTS
# ================================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("3D: Distribution Shift vs AUROC Drop\n"
             "H₀: ρ=0  |  H₁: Shift predicts failure",
             fontsize=13, fontweight="bold")

dist_names  = ["kl", "wasserstein", "frechet"]
dist_labels = ["KL Divergence", "Wasserstein Distance", "Fréchet Distance"]
palette     = plt.cm.Set1.colors

for ax, dt, dl in zip(axes, dist_names, dist_labels):
    sub = dist_df[dist_df["dist_type"] == dt].dropna()
    if sub.empty:
        continue

    rho, p_val = scipy_stats.spearmanr(sub["distance"], sub["auroc_drop"])

    for i, det in enumerate(sub["Detector"].unique()):
        ss = sub[sub["Detector"] == det]
        ax.scatter(ss["distance"], ss["auroc_drop"],
                   label=det, alpha=0.7, s=80,
                   color=palette[i % len(palette)])

    # Regression line
    x = sub["distance"].values
    y = sub["auroc_drop"].values
    m_r, b_r, _, _, _ = scipy_stats.linregress(x, y)
    xline = np.linspace(x.min(), x.max(), 100)
    ax.plot(xline, m_r * xline + b_r, "k-", linewidth=2, alpha=0.6)

    # 95% CI band via bootstrap
    boot_lines = []
    rng = np.random.default_rng(SEED)
    for _ in range(500):
        idx = rng.integers(0, len(x), size=len(x))
        mr, br, *_ = scipy_stats.linregress(x[idx], y[idx])
        boot_lines.append(mr * xline + br)
    lo_band = np.percentile(boot_lines, 2.5,  axis=0)
    hi_band = np.percentile(boot_lines, 97.5, axis=0)
    ax.fill_between(xline, lo_band, hi_band, alpha=0.15, color="grey")

    ax.axhline(0, color="k", linestyle="--", alpha=0.3)
    ax.set_xlabel(dl, fontsize=11)
    ax.set_ylabel("AUROC Drop", fontsize=11)
    sig = ("***" if p_val < 0.001 else
           "**"  if p_val < 0.01  else
           "*"   if p_val < 0.05  else "ns")
    ax.set_title(f"{dl}\nSpearman ρ={rho:.3f}  p={p_val:.4f}  {sig}",
                 fontsize=11, fontweight="bold")
    ax.legend(fontsize=7, bbox_to_anchor=(1.01, 1), loc="upper left")
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("./stage3_results/3D/shift_vs_drop.png",
            dpi=150, bbox_inches="tight")
plt.show()

print("\n" + "="*60)
print("SPEARMAN CORRELATION SUMMARY")
print("="*60)
for dt in dist_names:
    sub = dist_df[dist_df["dist_type"] == dt].dropna()
    if sub.empty:
        continue
    rho, p = scipy_stats.spearmanr(sub["distance"], sub["auroc_drop"])
    sig = ("✅ STRONG"   if abs(rho) > 0.7 else
           "⚠️  MODERATE" if abs(rho) > 0.4 else
           "❌ WEAK")
    print(f"  {dt:12s}  ρ={rho:+.3f}  p={p:.4f}  → {sig}")


# ================================================================
# CELL: DISTANCE RANKING PLOT
# ================================================================

dist_summary = pd.DataFrame(dist_results).T.reset_index()
dist_summary.columns = ["LLM", "KL", "Wasserstein", "Fréchet"]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, col in zip(axes, ["KL", "Wasserstein", "Fréchet"]):
    sub     = dist_summary.sort_values(col)
    colors_ = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, len(sub)))
    ax.barh(sub["LLM"], sub[col], color=colors_)
    ax.set_xlabel(col + " Distance")
    ax.set_title(f"{col} from Training Distribution")
    ax.grid(axis="x", alpha=0.3)

fig.suptitle("3D: Distance from HC3/ChatGPT Training Distribution",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("./stage3_results/3D/distance_ranking.png", dpi=150)
plt.show()